# Notebook 7: Randomized Search for Hyperparameter Tuning

**Difficulty:** ⭐⭐⭐ | **Estimated Time:** 2 hours  
**Week 8 — Regularization, Hyperparameter Tuning & Optimization**

---

## Why This Matters

Every ML model has hyperparameters — knobs you set *before* training. Choosing bad ones means a bad model, no matter how clean your data is. Grid search exhaustively tests every combination, which becomes impossibly expensive in high dimensions. **Randomized Search** gives you near-optimal results at a fraction of the cost.

---

## The Restaurant Analogy

Imagine you've just moved to a city with 10,000 restaurants. You want the best one.

- **Grid Search approach:** Divide the city into a systematic grid. Visit every restaurant on every block, row by row. Guaranteed to find the best — but it takes **years**.
- **Random Search approach:** Randomly pick 50 restaurants scattered across the entire city. You'll almost certainly find an excellent one without visiting every block.

Here's the deeper insight: most of what makes a restaurant *great* comes from a **few key factors** (chef quality, freshness of ingredients) — not every possible attribute. Random search naturally spreads out across these important dimensions, while grid search wastes time testing fine-grained variations of factors that barely matter.

---

## Plain English Explanation

**Grid Search:** Define a discrete set of values for each hyperparameter. Try every combination. If you have 5 hyperparameters × 10 values each → 100,000 model fits. This doesn't scale.

**Random Search:** Instead of a grid, sample hyperparameter values **randomly** from distributions. With 200 random samples you get far better coverage of the important hyperparameters than 200 grid points ever could.

**Key insight (Bergstra & Bengio, 2012):** In high-dimensional spaces, most hyperparameters have low importance. Grid search wastes evaluations testing many values of unimportant parameters. Random search gives each evaluation a unique value of *every* parameter.

---

## The Math Behind the Advantage

Suppose you have 2 hyperparameters: `alpha` (important) and `gamma` (unimportant).  
You have a budget of **25 evaluations**.

- **Grid search** (5×5): tests 5 unique values of `alpha` and 5 unique values of `gamma`
- **Random search** (25 samples): tests ~25 unique values of `alpha` and ~25 unique values of `gamma`

Random search covers the important `alpha` dimension **5× more densely** with the same budget!

Also: grid search must **discretize** continuous parameters, which introduces artificial gaps:  
$$\alpha \in \{0.001, 0.01, 0.1, 1.0, 10.0\}$$

Random search samples **continuously**:  
$$\alpha \sim \text{LogUniform}(0.001, 100)$$

The optimal value might be $\alpha = 0.037$ — grid search would never find it.

In [ ]:
# ============================================================
# Cell 1: Imports and global settings
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import ElasticNet, Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, KFold, GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from scipy.stats import loguniform, uniform, randint

# Reproducibility
np.random.seed(42)

# Plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
FIGSIZE = (12, 5)

print("Libraries loaded successfully.")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

In [ ]:
# ============================================================
# Cell 2: Build synthetic house price dataset
# California-style: 500 samples, 20 features
# Only ~5 features are truly predictive (the rest are noise)
# ============================================================
np.random.seed(42)

N_SAMPLES = 500
N_FEATURES = 20
N_INFORMATIVE = 5  # Only 5 of 20 features matter — rest are noise

# Generate feature matrix
X_raw = np.random.randn(N_SAMPLES, N_FEATURES)

# True coefficients: first 5 features are informative, rest are zero
true_coef = np.zeros(N_FEATURES)
true_coef[:N_INFORMATIVE] = [150_000, 80_000, -30_000, 50_000, 120_000]

# Generate target: house prices in USD
noise = np.random.randn(N_SAMPLES) * 25_000
y_raw = 450_000 + X_raw @ true_coef + noise

# Create informative column names
feature_names = (
    ['sq_feet', 'num_rooms', 'age_years', 'distance_cbd', 'school_rating']
    + [f'noise_feature_{i}' for i in range(1, N_FEATURES - N_INFORMATIVE + 1)]
)
df = pd.DataFrame(X_raw, columns=feature_names)
df['price'] = y_raw

# Train/test split (manual)
split_idx = int(0.8 * N_SAMPLES)
X_train = df[feature_names].values[:split_idx]
X_test  = df[feature_names].values[split_idx:]
y_train = df['price'].values[:split_idx]
y_test  = df['price'].values[split_idx:]

# Scale features (required for regularized models)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Training set : {X_train_sc.shape}")
print(f"Test set     : {X_test_sc.shape}")
print(f"Price range  : ${y_train.min():,.0f} — ${y_train.max():,.0f}")
print(f"\nInformative features : {feature_names[:N_INFORMATIVE]}")
print(f"Noise features       : {N_FEATURES - N_INFORMATIVE} columns of pure random noise")

## Section 1: Visual Proof — Why Random Search Beats Grid Search

Let's make the Bergstra & Bengio argument concrete with a picture.  
Suppose we're tuning two hyperparameters:
- `alpha` (regularization strength) — **important**, strongly affects model quality
- `gamma` (some less important param) — **unimportant**, barely affects model quality

With a budget of 25 evaluations, compare how many **unique values of alpha** each method samples.

In [ ]:
# ============================================================
# Cell 3: Visual demonstration — Grid vs Random coverage
# Shows why random search is superior in high dimensions
# ============================================================
np.random.seed(42)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Grid search: 5x5 = 25 points ---
grid_alpha = np.linspace(0, 1, 5)
grid_gamma = np.linspace(0, 1, 5)
gA, gG = np.meshgrid(grid_alpha, grid_gamma)
grid_points_alpha = gA.ravel()
grid_points_gamma = gG.ravel()

axes[0].scatter(grid_points_alpha, grid_points_gamma,
                s=100, color='steelblue', edgecolors='navy', linewidths=1.5, zorder=3)
axes[0].set_title('Grid Search (5×5 = 25 evaluations)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('alpha  ← IMPORTANT hyperparameter →', fontsize=12)
axes[0].set_ylabel('gamma  ← unimportant hyperparameter →', fontsize=12)
# Mark unique alpha values with vertical lines
for a in grid_alpha:
    axes[0].axvline(a, color='steelblue', alpha=0.25, linestyle='--')
axes[0].text(0.5, 1.04,
    f'Unique alpha values: {len(np.unique(grid_points_alpha))} / 25 evaluations',
    ha='center', transform=axes[0].transAxes, fontsize=11,
    color='steelblue', fontweight='bold')

# --- Random search: 25 random points ---
rand_alpha = np.random.uniform(0, 1, 25)
rand_gamma = np.random.uniform(0, 1, 25)

axes[1].scatter(rand_alpha, rand_gamma,
                s=100, color='tomato', edgecolors='darkred', linewidths=1.5, zorder=3)
axes[1].set_title('Random Search (25 random evaluations)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('alpha  ← IMPORTANT hyperparameter →', fontsize=12)
axes[1].set_ylabel('gamma  ← unimportant hyperparameter →', fontsize=12)
axes[1].text(0.5, 1.04,
    f'Unique alpha values: {len(np.unique(rand_alpha))} / 25 evaluations',
    ha='center', transform=axes[1].transAxes, fontsize=11,
    color='tomato', fontweight='bold')

plt.suptitle('Same Budget (25 evaluations): Grid vs Random Coverage',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("KEY INSIGHT:")
print(f"  Grid search  → {len(np.unique(grid_points_alpha))} unique alpha values")
print(f"  Random search→ {len(np.unique(rand_alpha))} unique alpha values")
print("  Random search explores the important dimension 5x more thoroughly!")

## Section 2: Continuous Distributions — The Right Way to Sample

Grid search is forced to **discretize** continuous hyperparameters. You must pick a finite set of values before knowing where the optimum lies.

Random search can sample from **continuous probability distributions** — much better coverage, and you don't need to pre-commit to specific values.

### Why log-uniform for alpha?

Regularization strength `alpha` spans **orders of magnitude**: 0.0001, 0.001, 0.01, 0.1, 1, 10, 100.  
A uniform distribution on [0.0001, 100] would almost never sample values below 1.  
A **log-uniform** distribution treats each order of magnitude equally — that's what we want.

In [ ]:
# ============================================================
# Cell 4: Visualize sampling distributions
# Log-uniform vs Uniform for alpha — why it matters
# ============================================================
np.random.seed(42)

n_samples = 1000

# Continuous distributions used in random search
alpha_loguniform = loguniform(1e-4, 100).rvs(n_samples)   # spans 6 orders of magnitude
alpha_uniform    = uniform(1e-4, 100).rvs(n_samples)      # almost always near 50
l1_ratio_uniform = uniform(0, 1).rvs(n_samples)           # between 0 (Ridge) and 1 (Lasso)
n_features_randint = randint(1, 20).rvs(n_samples)        # integer: 1 to 19

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Log-uniform alpha
axes[0].hist(np.log10(alpha_loguniform), bins=40, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_xlabel('log10(alpha)', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('alpha ~ LogUniform(1e-4, 100)\nEven coverage across orders of magnitude', fontsize=11)
axes[0].set_xticks([-4, -3, -2, -1, 0, 1, 2])
axes[0].set_xticklabels(['1e-4', '1e-3', '0.01', '0.1', '1', '10', '100'])

# Uniform alpha (bad choice)
axes[1].hist(np.log10(alpha_uniform), bins=40, color='tomato', edgecolor='white', alpha=0.85)
axes[1].set_xlabel('log10(alpha)', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_title('alpha ~ Uniform(1e-4, 100)\nAlmost always samples near 50 — bad!', fontsize=11)
axes[1].set_xticks([-4, -3, -2, -1, 0, 1, 2])
axes[1].set_xticklabels(['1e-4', '1e-3', '0.01', '0.1', '1', '10', '100'])

# l1_ratio uniform
axes[2].hist(l1_ratio_uniform, bins=30, color='mediumseagreen', edgecolor='white', alpha=0.85)
axes[2].set_xlabel('l1_ratio', fontsize=12)
axes[2].set_ylabel('Count', fontsize=12)
axes[2].set_title('l1_ratio ~ Uniform(0, 1)\nPerfect: covers [0=Ridge, 1=Lasso] evenly', fontsize=11)

plt.suptitle('Choosing the Right Distribution for Each Hyperparameter', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 3: Implement Randomized Search from Scratch

Before using sklearn's `RandomizedSearchCV`, let's build it ourselves so we understand every step.  
The algorithm is simple:

```
for i in range(n_iter):
    1. Sample one value per hyperparameter from its distribution
    2. Fit the model with those hyperparameters using cross-validation
    3. Record the CV score
    4. If this is the best score so far, save the params
return best_params, all_results
```

In [ ]:
# ============================================================
# Cell 5: Randomized Search from scratch
# Transparent implementation — no sklearn magic
# ============================================================
from sklearn.model_selection import cross_val_score

def randomized_search_cv(model_class, param_distributions, X, y,
                          n_iter=50, cv=5, scoring='r2', random_state=42):
    """
    Randomized hyperparameter search with cross-validation.

    Parameters
    ----------
    model_class       : sklearn estimator class (not instance)
    param_distributions : dict mapping param name -> scipy distribution or list
    X, y              : data
    n_iter            : number of random configurations to try
    cv                : number of cross-validation folds
    scoring           : sklearn scoring string

    Returns
    -------
    best_params  : dict of best hyperparameter values found
    best_score   : float — best mean CV score
    results_df   : DataFrame with all trials and their scores
    """
    np.random.seed(random_state)
    best_score = -np.inf
    best_params = None
    results = []

    for i in range(n_iter):
        # Step 1: Sample one value per hyperparameter
        params = {}
        for param_name, dist in param_distributions.items():
            if hasattr(dist, 'rvs'):          # scipy distribution object
                params[param_name] = dist.rvs()
            else:                              # plain Python list
                params[param_name] = np.random.choice(dist)

        # Step 2: Build model and evaluate with cross-validation
        try:
            model = model_class(**params)
            scores = cross_val_score(model, X, y, cv=cv, scoring=scoring)
            mean_score = scores.mean()
            std_score  = scores.std()
        except Exception:
            mean_score, std_score = -np.inf, np.inf

        # Step 3: Record result
        row = {**params, 'mean_cv_score': mean_score, 'std_cv_score': std_score, 'iteration': i}
        results.append(row)

        # Step 4: Update best
        if mean_score > best_score:
            best_score  = mean_score
            best_params = params.copy()

    return best_params, best_score, pd.DataFrame(results)

print("randomized_search_cv defined successfully.")
print("This is essentially what sklearn's RandomizedSearchCV does internally.")

In [ ]:
# ============================================================
# Cell 6: Run our custom randomized search on ElasticNet
# Searching over alpha (log-uniform) and l1_ratio (uniform)
# ============================================================
np.random.seed(42)

# Define the hyperparameter distributions
# alpha  : regularization strength — log-uniform over 6 orders of magnitude
# l1_ratio: 0 = pure Ridge, 1 = pure Lasso, in between = ElasticNet
param_distributions = {
    'alpha'   : loguniform(1e-3, 100),   # sample alpha from [0.001, 100] log-scale
    'l1_ratio': uniform(0.0, 1.0),       # sample l1_ratio from [0, 1] uniformly
}

# Run 50 random trials with 5-fold CV
best_params, best_score, results_df = randomized_search_cv(
    model_class=ElasticNet,
    param_distributions=param_distributions,
    X=X_train_sc,
    y=y_train,
    n_iter=50,
    cv=5,
    scoring='r2',
    random_state=42
)

print("=" * 50)
print("CUSTOM RANDOM SEARCH RESULTS")
print("=" * 50)
print(f"Best alpha    : {best_params['alpha']:.6f}")
print(f"Best l1_ratio : {best_params['l1_ratio']:.6f}")
print(f"Best R² score : {best_score:.6f}")
print()
print("Top 5 trials:")
top5 = results_df.nlargest(5, 'mean_cv_score')[['alpha', 'l1_ratio', 'mean_cv_score', 'std_cv_score']]
print(top5.to_string(index=False))

In [ ]:
# ============================================================
# Cell 7: Visualize the search — where did trials land?
# Color = CV score, so we can see the response surface
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# --- Left: scatter of all trials colored by score ---
scatter = axes[0].scatter(
    np.log10(results_df['alpha']),
    results_df['l1_ratio'],
    c=results_df['mean_cv_score'],
    cmap='RdYlGn',
    s=120,
    alpha=0.85,
    edgecolors='grey',
    linewidths=0.5
)
plt.colorbar(scatter, ax=axes[0], label='Mean CV R² Score')

# Star on best params
axes[0].scatter(
    np.log10(best_params['alpha']),
    best_params['l1_ratio'],
    marker='*', s=400, color='gold', edgecolors='black', linewidths=1.5, zorder=5,
    label=f"Best (R²={best_score:.4f})"
)
axes[0].set_xlabel('log10(alpha)', fontsize=12)
axes[0].set_ylabel('l1_ratio', fontsize=12)
axes[0].set_title('50 Random Trials — Color = CV R² Score', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=11)

# --- Right: Histogram of scores across trials ---
axes[1].hist(results_df['mean_cv_score'], bins=20, color='steelblue', edgecolor='white', alpha=0.85)
axes[1].axvline(best_score, color='gold', linewidth=2.5, linestyle='--',
                label=f'Best score = {best_score:.4f}')
axes[1].axvline(results_df['mean_cv_score'].mean(), color='tomato', linewidth=2, linestyle=':',
                label=f'Mean score = {results_df["mean_cv_score"].mean():.4f}')
axes[1].set_xlabel('Mean CV R² Score', fontsize=12)
axes[1].set_ylabel('Number of Trials', fontsize=12)
axes[1].set_title('Distribution of CV Scores Across Trials', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=11)

plt.suptitle('Random Search Results: ElasticNet on House Price Data',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 4: Convergence — Best Score vs Number of Evaluations

One of the most useful properties of random search: **the best score found so far improves quickly at first, then plateaus**.  
This means you can often stop early and still get a great result.

In [ ]:
# ============================================================
# Cell 8: Convergence curve — best score vs evaluations
# Shows how quickly random search finds a good solution
# ============================================================
np.random.seed(42)

# Run with more iterations to see the convergence shape
_, _, results_100 = randomized_search_cv(
    ElasticNet, param_distributions, X_train_sc, y_train,
    n_iter=100, cv=5, scoring='r2', random_state=42
)

# Compute running best (best score seen so far after k evaluations)
running_best = results_100['mean_cv_score'].cummax()

fig, ax = plt.subplots(figsize=(12, 5))

# Individual trial scores
ax.scatter(results_100['iteration'], results_100['mean_cv_score'],
           alpha=0.4, s=50, color='steelblue', label='Individual trial score', zorder=2)

# Running best (the important line)
ax.plot(results_100['iteration'], running_best,
        color='tomato', linewidth=3, label='Best score so far', zorder=3)

# Annotate the plateau
plateau_start = np.where(running_best.values >= running_best.values[-1] * 0.999)[0][0]
ax.axvline(plateau_start, color='gold', linestyle='--', linewidth=2, label=f'Plateau at trial {plateau_start}')

ax.set_xlabel('Number of Evaluations', fontsize=12)
ax.set_ylabel('Cross-Validation R² Score', fontsize=12)
ax.set_title('Convergence of Random Search: Best Score vs Evaluations',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

print(f"Score after  10 trials : {running_best.iloc[9]:.6f}")
print(f"Score after  25 trials : {running_best.iloc[24]:.6f}")
print(f"Score after  50 trials : {running_best.iloc[49]:.6f}")
print(f"Score after 100 trials : {running_best.iloc[99]:.6f}")
print(f"\nImprovement from trial 25 to 100: {running_best.iloc[99] - running_best.iloc[24]:.6f}")

## Section 5: Budget Experiment — Grid Search vs Random Search

Now for the key experiment: given the **exact same number of model evaluations**, which method finds better hyperparameters?

We'll compare:
- Grid search with N=25 (5×5), N=100 (10×10) points
- Random search with N=25, N=100 random samples

And we run each setup **30 times** to get distributions, not just single runs.

In [ ]:
# ============================================================
# Cell 9: Budget experiment
# Grid search vs Random search at N=25 and N=100 evaluations
# ============================================================
from sklearn.model_selection import GridSearchCV
np.random.seed(42)

N_REPEATS = 15  # How many independent runs per configuration
results_budget = {}

# === Grid search at N=25 (5x5) ===
grid_25 = {
    'alpha'   : np.logspace(-3, 2, 5),   # 5 values: 0.001, 0.01, ..., 100
    'l1_ratio': np.linspace(0, 1, 5)     # 5 values: 0.0, 0.25, 0.5, 0.75, 1.0
}
gs_25 = GridSearchCV(ElasticNet(), grid_25, cv=5, scoring='r2', n_jobs=-1)
gs_25.fit(X_train_sc, y_train)
results_budget['Grid N=25'] = [gs_25.best_score_] * N_REPEATS  # deterministic

# === Grid search at N=100 (10x10) ===
grid_100 = {
    'alpha'   : np.logspace(-3, 2, 10),
    'l1_ratio': np.linspace(0, 1, 10)
}
gs_100 = GridSearchCV(ElasticNet(), grid_100, cv=5, scoring='r2', n_jobs=-1)
gs_100.fit(X_train_sc, y_train)
results_budget['Grid N=100'] = [gs_100.best_score_] * N_REPEATS  # deterministic

# === Random search at N=25 (15 independent runs) ===
rand_25_scores = []
for seed in range(N_REPEATS):
    _, score, _ = randomized_search_cv(
        ElasticNet, param_distributions, X_train_sc, y_train,
        n_iter=25, cv=5, scoring='r2', random_state=seed
    )
    rand_25_scores.append(score)
results_budget['Random N=25'] = rand_25_scores

# === Random search at N=100 (15 independent runs) ===
rand_100_scores = []
for seed in range(N_REPEATS):
    _, score, _ = randomized_search_cv(
        ElasticNet, param_distributions, X_train_sc, y_train,
        n_iter=100, cv=5, scoring='r2', random_state=seed
    )
    rand_100_scores.append(score)
results_budget['Random N=100'] = rand_100_scores

# Summary table
print("Budget Comparison Results:")
print("-" * 60)
print(f"{'Method':<15} {'Mean R²':>10} {'Std R²':>10} {'Max R²':>10}")
print("-" * 60)
for name, scores in results_budget.items():
    print(f"{name:<15} {np.mean(scores):>10.6f} {np.std(scores):>10.6f} {np.max(scores):>10.6f}")

In [ ]:
# ============================================================
# Cell 10: Visualize budget experiment results
# Boxplot: compare score distributions across methods
# ============================================================
fig, ax = plt.subplots(figsize=(11, 6))

labels  = list(results_budget.keys())
scores  = list(results_budget.values())
colors  = ['steelblue', 'royalblue', 'tomato', 'darkred']

bp = ax.boxplot(scores, patch_artist=True, notch=False, widths=0.5)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

# Overlay individual points
for i, (score_list, color) in enumerate(zip(scores, colors), start=1):
    jitter = np.random.normal(i, 0.06, size=len(score_list))
    ax.scatter(jitter, score_list, alpha=0.6, s=50, color=color, edgecolors='black', linewidths=0.5)

ax.set_xticklabels(labels, fontsize=12)
ax.set_ylabel('Best CV R² Score Found', fontsize=12)
ax.set_title('Grid Search vs Random Search — Same Evaluation Budget',
             fontsize=13, fontweight='bold')
ax.set_ylim(bottom=ax.get_ylim()[0] - 0.01)

# Annotation
ax.axhline(np.mean(results_budget['Grid N=100']), color='royalblue',
           linestyle='--', alpha=0.6, linewidth=1.5, label='Grid N=100 baseline')
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

print("CONCLUSION:")
print("Random search at N=25 often matches or beats Grid search at N=25.")
print("Random search samples continuous values — no gaps in coverage.")

## Section 6: Verify with sklearn's RandomizedSearchCV

Now let's compare our hand-rolled implementation with sklearn's built-in `RandomizedSearchCV`.  
They should find very similar (not identical — different random seeds) best parameters.

In [ ]:
# ============================================================
# Cell 11: Sklearn RandomizedSearchCV — official implementation
# Compare with our custom version
# ============================================================
from sklearn.model_selection import RandomizedSearchCV
np.random.seed(42)

# Same distributions, same budget, same cv
sklearn_param_dist = {
    'alpha'   : loguniform(1e-3, 100),
    'l1_ratio': uniform(0.0, 1.0),
}

sklearn_rs = RandomizedSearchCV(
    estimator=ElasticNet(),
    param_distributions=sklearn_param_dist,
    n_iter=50,
    cv=5,
    scoring='r2',
    random_state=42,
    n_jobs=-1,
    verbose=0
)
sklearn_rs.fit(X_train_sc, y_train)

print("=" * 55)
print("COMPARISON: Custom vs Sklearn RandomizedSearchCV")
print("=" * 55)
print()
print("CUSTOM IMPLEMENTATION:")
print(f"  Best alpha    : {best_params['alpha']:.6f}")
print(f"  Best l1_ratio : {best_params['l1_ratio']:.6f}")
print(f"  Best R²       : {best_score:.6f}")
print()
print("SKLEARN RandomizedSearchCV:")
print(f"  Best alpha    : {sklearn_rs.best_params_['alpha']:.6f}")
print(f"  Best l1_ratio : {sklearn_rs.best_params_['l1_ratio']:.6f}")
print(f"  Best R²       : {sklearn_rs.best_score_:.6f}")
print()
print("NOTE: Params differ slightly because of different random seeds,")
print("but R² scores should be very close — both found good solutions.")

In [ ]:
# ============================================================
# Cell 12: Final model — fit with best params, evaluate on test
# ============================================================
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
np.random.seed(42)

# Use sklearn's best params (more robust due to more optimized implementation)
final_model = ElasticNet(
    alpha=sklearn_rs.best_params_['alpha'],
    l1_ratio=sklearn_rs.best_params_['l1_ratio'],
    max_iter=10000
)
final_model.fit(X_train_sc, y_train)
y_pred = final_model.predict(X_test_sc)

# Compute test metrics
r2  = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("=" * 45)
print("FINAL MODEL TEST SET PERFORMANCE")
print("=" * 45)
print(f"R²   : {r2:.4f}")
print(f"MAE  : ${mae:>10,.0f}")
print(f"RMSE : ${rmse:>10,.0f}")

# Visualize predictions vs actuals
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(y_test, y_pred, alpha=0.6, s=60, color='steelblue', edgecolors='navy', linewidths=0.5)
lims = [min(y_test.min(), y_pred.min()) - 10000, max(y_test.max(), y_pred.max()) + 10000]
ax.plot(lims, lims, 'r--', linewidth=2, label='Perfect prediction')
ax.set_xlabel('Actual Price ($)', fontsize=12)
ax.set_ylabel('Predicted Price ($)', fontsize=12)
ax.set_title(f'Predicted vs Actual House Prices\nR² = {r2:.4f} | MAE = ${mae:,.0f}',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

## Section 7: When to Use Which — Decision Guide

Here is a practical comparison across five decision dimensions:

In [ ]:
# ============================================================
# Cell 13: Grid Search vs Random Search — comparison table
# When to use which method
# ============================================================
comparison_data = {
    'Dimension': [
        '# Hyperparameters',
        'Hyperparameter type',
        'Compute budget',
        'Reproducibility',
        'Parameter importance'
    ],
    'Grid Search': [
        '1–3 (combinatorial explosion beyond this)',
        'Discrete only (must hand-pick values)',
        'High: evaluates ALL combinations',
        'Fully deterministic — always same result',
        'Tests same values for all params equally'
    ],
    'Random Search': [
        '3–20+ (scales gracefully)',
        'Continuous or discrete distributions',
        'Controlled: set n_iter to match budget',
        'Stochastic — set random_state for reproducibility',
        'Naturally more coverage on important dims'
    ],
    'Winner': [
        'Random (unless <= 2 params)',
        'Random (continuous sampling)',
        'Random (fixed budget)',
        'Tie (both can be reproducible)',
        'Random (Bergstra & Bengio 2012)'
    ]
}

comparison_df = pd.DataFrame(comparison_data)

# Pretty print
print("GRID SEARCH vs RANDOM SEARCH — PRACTICAL COMPARISON")
print("=" * 100)
for _, row in comparison_df.iterrows():
    print(f"\n[{row['Dimension']}]")
    print(f"  Grid   : {row['Grid Search']}")
    print(f"  Random : {row['Random Search']}")
    print(f"  Winner : >>> {row['Winner']} <<<")

print()
print("PRACTICAL RULE: Start with random search unless you have <= 2 hyperparameters")
print("and they are both truly discrete. Then try Bayesian optimization (Notebook 8).")

In [ ]:
# ============================================================
# Cell 14: Extended example — Ridge with more hyperparameters
# Showing random search scales to higher dimensions
# ============================================================
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
np.random.seed(42)

# Pipeline: polynomial features + Ridge regression
# 3 hyperparameters: alpha, degree, fit_intercept
pipeline = Pipeline([
    ('poly', PolynomialFeatures()),
    ('ridge', Ridge())
])

# 3D hyperparameter space — grid search would be e.g. 10x3x2 = 60 combinations
param_dist_3d = {
    'poly__degree'          : [1, 2],           # polynomial degree (list = random choice)
    'ridge__alpha'          : loguniform(1e-3, 1000),
    'ridge__fit_intercept'  : [True, False],
}

rs_pipeline = RandomizedSearchCV(
    pipeline, param_dist_3d,
    n_iter=40, cv=5, scoring='r2',
    random_state=42, n_jobs=-1
)
rs_pipeline.fit(X_train_sc, y_train)

print("Ridge + Polynomial Features (3D hyperparameter space)")
print("=" * 55)
print(f"Best params: {rs_pipeline.best_params_}")
print(f"Best CV R² : {rs_pipeline.best_score_:.6f}")

# Test score
y_pred_pipe = rs_pipeline.predict(X_test_sc)
print(f"Test R²    : {r2_score(y_test, y_pred_pipe):.6f}")

In [ ]:
# ============================================================
# Cell 15: Multi-budget convergence comparison
# Show how score improves as budget increases for random search
# ============================================================
np.random.seed(42)

budgets = [5, 10, 15, 20, 25, 30, 40, 50, 75, 100]
n_repeats = 10
mean_scores = []
std_scores  = []

for budget in budgets:
    scores_at_budget = []
    for seed in range(n_repeats):
        _, score, _ = randomized_search_cv(
            ElasticNet, param_distributions, X_train_sc, y_train,
            n_iter=budget, cv=5, scoring='r2', random_state=seed * 7
        )
        scores_at_budget.append(score)
    mean_scores.append(np.mean(scores_at_budget))
    std_scores.append(np.std(scores_at_budget))

mean_scores = np.array(mean_scores)
std_scores  = np.array(std_scores)

# Grid search benchmarks (deterministic)
gs_n25_score  = gs_25.best_score_
gs_n100_score = gs_100.best_score_

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(budgets, mean_scores, 'o-', color='tomato', linewidth=2.5,
        markersize=8, label='Random Search (mean over 10 runs)')
ax.fill_between(budgets, mean_scores - std_scores, mean_scores + std_scores,
                alpha=0.25, color='tomato', label='±1 std dev')
ax.axhline(gs_n25_score, color='steelblue', linestyle='--', linewidth=2,
           label=f'Grid N=25 (5×5): {gs_n25_score:.4f}')
ax.axhline(gs_n100_score, color='royalblue', linestyle='-.', linewidth=2,
           label=f'Grid N=100 (10×10): {gs_n100_score:.4f}')
ax.set_xlabel('Random Search Budget (n_iter)', fontsize=12)
ax.set_ylabel('Best CV R² Score', fontsize=12)
ax.set_title('Random Search: How Score Improves with Budget',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

## Self-Check Questions

Test your understanding. Try answering before looking at the answers.

---

**Question 1:** You're tuning a model with 5 hyperparameters, each with 10 possible values. Grid search requires 10⁵ = 100,000 model fits. Random search with n_iter=200 evaluates only 200. Yet Bergstra & Bengio (2012) claim random search is likely to find *better* hyperparameters. Why?

<details>
<summary>Click to reveal answer</summary>

**Answer:** The key insight is that in most ML problems, **not all hyperparameters are equally important**. Only 1–2 hyperparameters typically have a large effect on model performance; the others barely matter.

With grid search over 5 hyperparameters × 10 values, each unique value of the *important* hyperparameter is paired with a fixed set of values for all other parameters — but you test those same important values only 10,000 times (the number of combos of the other 4 params), giving you just 10 unique important-parameter settings.

With 200 random trials, each trial independently samples ALL hyperparameters, including the important one. You get ~200 distinct values of the important hyperparameter — far better coverage at a fraction of the cost.

Additionally, grid search **discretizes** the important parameter into 10 fixed values. The true optimum might lie between grid points. Random search samples continuously, so it can discover values no grid would ever test.

</details>

---

**Question 2:** You're searching for the regularization strength `alpha` in ElasticNet over the range [0.0001, 100]. Why should you sample from a **log-uniform** distribution rather than a **uniform** distribution?

<details>
<summary>Click to reveal answer</summary>

**Answer:** Regularization strength operates on a **multiplicative scale** (orders of magnitude), not an additive one. The difference between α=0.001 and α=0.01 is just as meaningful as the difference between α=1 and α=10 — both represent a 10× change.

If you sample from `Uniform(0.0001, 100)`, the probability of drawing a value below 1 is only 1% (since 1/100 of the range is [0, 1]). You would almost never explore small values of alpha, even though that entire region of the search space matters.

Log-uniform sampling (i.e., sample `log(alpha)` uniformly, then exponentiate) gives **equal probability mass to each order of magnitude**: [0.0001, 0.001], [0.001, 0.01], [0.01, 0.1], [0.1, 1], [1, 10], [10, 100]. This matches how we actually reason about regularization.

In scipy: `loguniform(1e-4, 100).rvs()` does exactly this.

</details>

---

**Question 3:** Random search has inherent randomness — two independent runs can give different best parameters. Is this a fundamental flaw? How do you handle it in practice?

<details>
<summary>Click to reveal answer</summary>

**Answer:** It is **not a fundamental flaw** — it is a manageable property. Here's why it is not problematic and how to handle it:

1. **Set `random_state`:** sklearn's `RandomizedSearchCV` accepts `random_state=42`. With the same seed, you always get the same result. This handles reproducibility for reporting and deployment.

2. **The variance is small in practice:** After 50–100 trials, random search has explored enough of the space that different runs converge to very similar performance (even if the exact parameter values differ slightly). The *scores* are stable even if exact params vary.

3. **Run multiple times and take the best:** If you have compute to spare, run random search 3–5 times with different seeds and keep the best result overall. This is cheap compared to grid search.

4. **Contrast with grid search:** Grid search is deterministic but brittle in a different way — it only tests the exact grid points you pre-specify. If the optimum lies between grid points, you will *consistently* miss it. Random search's stochasticity is actually more likely to accidentally land near the true optimum.

5. **In production:** always fix the random seed, document it, and retrain with the same seed for reproducibility.

</details>

In [ ]:
# ============================================================
# Cell 16: Summary — key takeaways from this notebook
# ============================================================
print("=" * 65)
print("NOTEBOOK 7 SUMMARY: RANDOMIZED SEARCH")
print("=" * 65)
print()
print("1. CORE INSIGHT (Bergstra & Bengio 2012):")
print("   Most hyperparameters have low importance. Random search")
print("   gives more unique values of the important ones.")
print()
print("2. CONTINUOUS DISTRIBUTIONS:")
print("   - alpha (learning rate, regularization): loguniform")
print("   - ratios and probabilities: uniform(0, 1)")
print("   - integers (depth, estimators): randint")
print()
print("3. CONVERGENCE:")
print("   Best score improves quickly then plateaus.")
print("   Often 25–50 trials is enough for 2–3 hyperparameters.")
print()
print("4. BUDGET RULE OF THUMB:")
print("   n_iter = 60 × (number of hyperparameters) is a solid start.")
print()
print("5. NEXT STEP:")
print("   Bayesian optimization (Notebook 8) does even better by")
print("   learning from previous trials to choose the next one.")
print("   Cost: sequential (can't parallelize as easily).")
print()
print("WHAT YOU BUILT:")
print("  - randomized_search_cv() from scratch")
print("  - Budget comparison: grid vs random at N=25 and N=100")
print("  - Convergence curves and score distribution plots")
print("  - Verified results against sklearn RandomizedSearchCV")